# 成员6：正式版 DEA + K-Means 建模 Notebook（终修版）

本 Notebook 用于完成方向二的正式建模任务，核心目标包括：

1. 读取 `Final_Model_Dataset_Strict.csv`
2. 对非期望产出做正向化处理
3. 逐年运行 DEA（CCR / BCC，输入导向）
4. 计算规模效率
5. 对 2022 年进行 K-Means 聚类
6. 自动生成与聚类画像一致的 4 类中文标签
7. 导出正式结果表与 Top / Bottom 排名表

> 说明：本版重点修正了两个问题：  
> - **排序逻辑**：前10 / 后10 现在严格按 `DEA_BCC → DEA_CCR → 规模效率` 排序  
> - **聚类命名逻辑**：不再手工写死，而是根据聚类画像自动命名


## 1. 导入库与设置参数

In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"E:\2026_BigData_Project")
DATA_PATH = BASE_DIR / "data" / "processed" / "Final_Model_Dataset_Strict.csv"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_LIST = [2020, 2021, 2022]
CLUSTER_YEAR = 2022
N_CLUSTERS = 4
RANDOM_SEED = 42

INPUT_COLS = [
    "每千人口床位数(张/千人)",
    "卫生技术人员总数(人)",
    "人均医疗卫生财政支出(元)",
]
BASE_OUTPUT_COLS = ["预期寿命 (岁)"]
UNDESIRABLE_COLS = ["孕产妇死亡率 (1/10 万)", "婴儿死亡率 (‰)"]

REGION_COL = "地区"
YEAR_COL = "年份"


## 2. 读取数据并完成非期望产出正向化

In [ ]:

def load_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required_cols = [REGION_COL, YEAR_COL] + INPUT_COLS + BASE_OUTPUT_COLS + UNDESIRABLE_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"缺少必要字段: {missing}")
    return df.copy()


def transform_undesirable_outputs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in UNDESIRABLE_COLS:
        new_col = f"inv_{col.split(' ')[0]}"
        out[new_col] = 1.0 / out[col].astype(float).replace(0, np.nan)
    return out


df = load_data(DATA_PATH)
df = transform_undesirable_outputs(df)

print(df.shape)
df.head(3)


## 3. DEA 求解函数

In [ ]:

def build_year_subset(df: pd.DataFrame, year: int) -> pd.DataFrame:
    sub = df[df[YEAR_COL] == year].copy()
    if sub.empty:
        raise ValueError(f"没有找到年份 {year} 的数据。")
    return sub.sort_values(REGION_COL).reset_index(drop=True)


def solve_dea_input_oriented(X: np.ndarray, Y: np.ndarray, rts: str = "bcc") -> np.ndarray:
    n, m = X.shape
    _, s = Y.shape
    eff_scores = np.zeros(n)

    for o in range(n):
        c = np.zeros(n + 1)
        c[-1] = 1.0

        A_ub, b_ub = [], []

        for i in range(m):
            row = np.zeros(n + 1)
            row[:n] = X[:, i]
            row[-1] = -X[o, i]
            A_ub.append(row)
            b_ub.append(0.0)

        for r in range(s):
            row = np.zeros(n + 1)
            row[:n] = -Y[:, r]
            A_ub.append(row)
            b_ub.append(-Y[o, r])

        A_eq, b_eq = None, None
        if rts.lower() == "bcc":
            A_eq = [np.r_[np.ones(n), 0.0]]
            b_eq = [1.0]

        bounds = [(0, None)] * n + [(0, 1)]

        res = linprog(
            c=c,
            A_ub=np.array(A_ub),
            b_ub=np.array(b_ub),
            A_eq=np.array(A_eq) if A_eq is not None else None,
            b_eq=np.array(b_eq) if b_eq is not None else None,
            bounds=bounds,
            method="highs",
        )

        eff_scores[o] = np.nan if not res.success else res.x[-1]

    return eff_scores


def run_dea_for_year(df_year: pd.DataFrame) -> pd.DataFrame:
    out = df_year.copy()
    output_cols = BASE_OUTPUT_COLS + ["inv_孕产妇死亡率", "inv_婴儿死亡率"]

    X = out[INPUT_COLS].astype(float).values
    Y = out[output_cols].astype(float).values

    X_scaled = X.copy()
    X_scaled[:, 1] = X_scaled[:, 1] / 10000.0
    X_scaled[:, 2] = X_scaled[:, 2] / 1000.0
    Y_scaled = Y.copy()

    out["DEA_CCR"] = solve_dea_input_oriented(X_scaled, Y_scaled, rts="ccr")
    out["DEA_BCC"] = solve_dea_input_oriented(X_scaled, Y_scaled, rts="bcc")
    out["规模效率"] = out["DEA_CCR"] / out["DEA_BCC"]
    return out


def run_dea_all_years(df: pd.DataFrame, year_list=None) -> pd.DataFrame:
    if year_list is None:
        year_list = sorted(df[YEAR_COL].unique().tolist())
    frames = []
    for year in year_list:
        frames.append(run_dea_for_year(build_year_subset(df, year)))
    return pd.concat(frames, ignore_index=True)


## 4. 运行 2020–2022 三年 DEA

In [ ]:

dea_by_year = run_dea_all_years(df, YEAR_LIST)
dea_by_year.head()


## 5. 构造聚类特征并进行 K-Means 聚类

In [ ]:

def build_cluster_features(df_2022: pd.DataFrame):
    result = df_2022.copy()

    feature_cols = INPUT_COLS + BASE_OUTPUT_COLS + ["DEA_BCC", "规模效率"]
    features = result[feature_cols].astype(float)

    scaler = StandardScaler()
    X_std = scaler.fit_transform(features)
    std_df = pd.DataFrame(X_std, columns=[f"{c}_z" for c in feature_cols], index=result.index)

    input_z_cols = [f"{c}_z" for c in INPUT_COLS]
    eff_z_cols = [f"{BASE_OUTPUT_COLS[0]}_z", "DEA_BCC_z", "规模效率_z"]

    result["投入综合均值_z"] = std_df[input_z_cols].mean(axis=1)
    result["效率综合均值_z"] = std_df[eff_z_cols].mean(axis=1)

    return result, features


def run_kmeans(df_2022: pd.DataFrame, n_clusters=4, random_state=42):
    result, features = build_cluster_features(df_2022)

    scaler = StandardScaler()
    X_std = scaler.fit_transform(features)

    model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=50)
    result["Cluster"] = model.fit_predict(X_std)

    return result


df_2022 = dea_by_year[dea_by_year[YEAR_COL] == CLUSTER_YEAR].copy()
clustered_2022 = run_kmeans(df_2022, n_clusters=N_CLUSTERS, random_state=RANDOM_SEED)
clustered_2022.head()


## 6. 根据聚类画像自动命名 4 个 Cluster

In [ ]:

def make_cluster_profile(df_clustered: pd.DataFrame) -> pd.DataFrame:
    profile = (
        df_clustered
        .groupby("Cluster", as_index=False)
        .agg({
            "投入综合均值_z": "mean",
            "效率综合均值_z": "mean",
            "DEA_BCC": "mean",
            "规模效率": "mean",
            REGION_COL: "count"
        })
        .rename(columns={REGION_COL: "省份数"})
        .sort_values("Cluster")
        .reset_index(drop=True)
    )
    return profile


def auto_name_clusters(profile: pd.DataFrame) -> dict:
    prof = profile.copy()
    cluster_names = {}

    c_high_eff = prof.sort_values("效率综合均值_z", ascending=False).iloc[0]["Cluster"]
    cluster_names[c_high_eff] = "高效率领先区"

    remaining = prof[~prof["Cluster"].isin(cluster_names.keys())].copy()

    c_low_eff_row = remaining.sort_values("效率综合均值_z", ascending=True).iloc[0]
    c_low_eff = c_low_eff_row["Cluster"]
    if c_low_eff_row["投入综合均值_z"] >= 0:
        cluster_names[c_low_eff] = "中高投入低转化洼地区"
    else:
        cluster_names[c_low_eff] = "低投入承压短板区"

    remaining = prof[~prof["Cluster"].isin(cluster_names.keys())].copy()

    c_low_input = remaining.sort_values("投入综合均值_z", ascending=True).iloc[0]["Cluster"]
    cluster_names[c_low_input] = "低投入约束型地区"

    remaining = prof[~prof["Cluster"].isin(cluster_names.keys())].copy()

    c_last = remaining.iloc[0]["Cluster"]
    cluster_names[c_last] = "中等效率平衡发展区"

    return cluster_names


def attach_cluster_names(df_clustered: pd.DataFrame):
    profile = make_cluster_profile(df_clustered)
    name_map = auto_name_clusters(profile)

    out = df_clustered.copy()
    out["Cluster_Name"] = out["Cluster"].map(name_map)

    profile["Cluster_Name"] = profile["Cluster"].map(name_map)
    profile = profile.sort_values(["投入综合均值_z", "效率综合均值_z"], ascending=[False, False]).reset_index(drop=True)

    return out, profile, name_map


clustered_2022, cluster_profile_2022, name_map = attach_cluster_names(clustered_2022)

cluster_profile_2022


## 7. 生成 Top10 / Bottom10 排名表

In [ ]:

def build_three_year_mean(df_by_year: pd.DataFrame) -> pd.DataFrame:
    return (
        df_by_year
        .groupby(REGION_COL, as_index=False)
        .agg({"DEA_CCR": "mean", "DEA_BCC": "mean", "规模效率": "mean"})
        .rename(columns={
            "DEA_CCR": "DEA_CCR_三年均值",
            "DEA_BCC": "DEA_BCC_三年均值",
            "规模效率": "规模效率_三年均值"
        })
    )


def build_top_bottom(df_2022_clustered: pd.DataFrame):
    sort_cols = ["DEA_BCC", "DEA_CCR", "规模效率"]
    ranked = df_2022_clustered.sort_values(by=sort_cols, ascending=[False, False, False]).reset_index(drop=True)
    top10 = ranked.head(10).copy()
    bottom10 = ranked.tail(10).sort_values(by=sort_cols, ascending=[True, True, True]).reset_index(drop=True)
    return top10, bottom10, ranked


top10_2022, bottom10_2022, ranked_2022 = build_top_bottom(clustered_2022)

top10_2022[[REGION_COL, "DEA_BCC", "DEA_CCR", "规模效率", "Cluster_Name"]]


## 8. 查看后10省份

In [ ]:

bottom10_2022[[REGION_COL, "DEA_BCC", "DEA_CCR", "规模效率", "Cluster_Name"]]


## 9. 导出正式结果文件

In [ ]:

def save_csv(df: pd.DataFrame, filename: str):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[OK] 已保存：{path}")


mean_3y = build_three_year_mean(dea_by_year)

save_csv(dea_by_year, "DEA_Scores_By_Year.csv")
save_csv(clustered_2022, "DEA_Scores_Clusters_Final.csv")
save_csv(mean_3y, "DEA_Scores_Clusters_3YearMean.csv")
save_csv(cluster_profile_2022, "Cluster_Profile_2022.csv")
save_csv(top10_2022, "DEA_BCC_Top10_2022.csv")
save_csv(bottom10_2022, "DEA_BCC_Bottom10_2022.csv")


## 10. 输出摘要结果

In [ ]:

summary = (
    dea_by_year.groupby(YEAR_COL)[["DEA_CCR", "DEA_BCC", "规模效率"]]
    .agg(["mean", "min", "max"])
    .round(6)
)
summary


## 11. 阶段性解释

本终修版重点解决了两类问题：

### （1）排序口径不一致
现在前10 / 后10 严格按以下顺序排序：
1. `DEA_BCC`
2. `DEA_CCR`
3. `规模效率`

因此表格标题与实际排序逻辑保持一致。

### （2）聚类标签与画像不匹配
现在的 `Cluster_Name` 不再手工固定，而是由以下逻辑自动决定：
- 效率综合最高：**高效率领先区**
- 效率综合最低且投入偏高：**中高投入低转化洼地区**
- 投入最低：**低投入约束型地区**
- 剩余一类：**中等效率平衡发展区**

这样得到的中文标签会与聚类画像更一致，更适合直接写入说明文档。
